## Purpose

Integrate the finalized occupational task-affordance configuration with the
occupation-level Eloundou aggregate GenAI exposure benchmark.

The task-affordance configuration and exposure benchmark are constructed
independently upstream. This notebook merges the two occupation-level datasets
by SOC6 to create the integrated configuration–exposure input used in subsequent
RQ1 analysis.

No upstream configuration or exposure measures are reconstructed in this notebook.

In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes-full"
)

HUMAN_GPT4_FILE = (
    BASE_DIR
    / "06_Eloundou_Task_Alignment"
    / "Output"
    / "Occupational_Eloundou_Exposure_Benchmark_v1.csv"
)

LAYER3_FILE = (
    BASE_DIR
    / "05_Layer3_Task_Configuration"
    / "Output"
    / "Occupational_Task_Affordance_Configuration_v6.csv"
)

human_gpt4_exposure = pd.read_csv(
    HUMAN_GPT4_FILE,
    dtype={"soc6": str}
)

layer3 = pd.read_csv(
    LAYER3_FILE,
    dtype={"soc6": str}
)

print("Configuration records:", len(layer3))
print("Configuration SOC6:", layer3["soc6"].nunique())

print("Exposure records:", len(human_gpt4_exposure))
print("Exposure SOC6:", human_gpt4_exposure["soc6"].nunique())

In [ ]:
# Step 2 — Validate and integrate configuration with exposure

exposure_columns = [
    "soc6",
    "matched_task_count",
    "total_task_weight",
    "human_alpha",
    "human_beta",
    "human_gamma",
    "gpt4_alpha",
    "gpt4_beta",
    "gpt4_gamma",
]

assert not layer3["soc6"].duplicated().any()
assert not human_gpt4_exposure["soc6"].duplicated().any()
assert set(layer3["soc6"]) == set(human_gpt4_exposure["soc6"])

rq1_config_exposure = layer3.merge(
    human_gpt4_exposure[exposure_columns],
    on="soc6",
    how="inner",
    validate="one_to_one"
)

assert len(rq1_config_exposure) == 771
assert rq1_config_exposure["soc6"].nunique() == 771
assert rq1_config_exposure[exposure_columns[1:]].notna().all().all()

print("Integrated records:", len(rq1_config_exposure))
print("Integrated SOC6:", rq1_config_exposure["soc6"].nunique())
print("Integrated columns:", rq1_config_exposure.shape[1])

In [ ]:
display(rq1_config_exposure.head())

In [ ]:
# Step 4 — Save canonical configuration–exposure dataset

RQ1_CONFIG_EXPOSURE = BASE_DIR / "07_GenAI_Exposure" / "Output"

RQ1_CONFIG_EXPOSURE_FILE = (
    BASE_DIR
    / "07_GenAI_Exposure"
    / "Output"
    / "Occupational_Configuration_Exposure_v1.csv"
)

rq1_config_exposure.to_csv(
    RQ1_CONFIG_EXPOSURE_FILE,
    index=False
)

print("Records saved:", len(rq1_config_exposure))
print("SOC6 saved:", rq1_config_exposure["soc6"].nunique())
print("Saved:", RQ1_CONFIG_EXPOSURE_FILE)